In [0]:
from pyspark.sql import functions as F

silver_transactions_df = spark.table("workspace.banking_silver.silver_transactions")

print("Silver transactions: ",silver_transactions_df.count())

In [0]:
silver_transactions_df.printSchema()

In [0]:
# Create transaction_date

gold_transactions_df = silver_transactions_df.withColumn(
    "transaction_date",
    F.to_date(F.col("transaction_timestamp"))
)

In [0]:
# Verify transaction_date
gold_transactions_df.select(
    "transaction_timestamp",
    "transaction_date"
).show(10)

In [0]:
# Check transaction status distribution
gold_transactions_df.groupBy("status")\
                    .count()\
                    .orderBy("status")\
                    .show()

In [0]:
gold_daily_transaction_summary = gold_transactions_df.groupBy(
    "transaction_date",
    "transaction_type",
    "status",
).agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
)

In [0]:
# Inspect the Gold dataset
gold_daily_transaction_summary.orderBy(
    "transaction_date",
    "transaction_type",
    "total_amount"
).show(50)

In [0]:
# Validate transaction count and amount
silver_totals = gold_transactions_df.agg(
    F.count("transaction_id").alias("total_count"),
    F.sum("amount").alias("total_amount")
).first()

gold_totals = gold_daily_transaction_summary.agg(
    F.count("transaction_count").alias("total_count"),
    F.sum("total_amount").alias("total_amount")
).first()

print("Silver Transaction Count:", silver_totals["total_count"])
print("Gold Transaction Count:", gold_totals["total_count"])

print("Silver Total Amount:", silver_totals["total_amount"])
print("Gold Total Amount:", gold_totals["total_amount"])

In [0]:
print(
    "Total transactions represented in Gold:",
    gold_daily_transaction_summary.agg(
        F.sum("transaction_count")
    ).first()[0]
)

In [0]:
gold_daily_transaction_summary.write\
                              .format("delta")\
                              .mode("overwrite")\
                              .saveAsTable("workspace.banking_gold.gold_daily_transaction_summary")

In [0]:
display(spark.table("workspace.banking_gold.gold_daily_transaction_summary").count())
display(spark.table("workspace.banking_gold.gold_daily_transaction_summary").show(50))

In [0]:
# Load the Silver Transactions table
silver_transactions_df = spark.table("workspace.banking_silver.silver_transactions")

print(silver_transactions_df.count())

In [0]:
# Check source-system distribution
silver_transactions_df.groupBy("source_system")\
                      .count()\
                      .orderBy("source_system")\
                      .show()

In [0]:
# Create transaction_date
gold_source_transactions_df = silver_transactions_df.withColumn(
    "transaction_date",
    F.to_date("transaction_timestamp")
)

In [0]:
gold_source_transactions_df.groupBy(
    "source_system","status"
).count().orderBy("source_system","status").show()

In [0]:
gold_daily_transaction_summary_by_source_df = gold_source_transactions_df.groupBy(
    "transaction_date",
    "source_system",
    "status"
).agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
)

In [0]:
gold_daily_transaction_summary_by_source_df.orderBy(
    "transaction_date",
    "source_system",
    "status"
).show(50)

In [0]:
gold_daily_transaction_summary_by_source_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
gold_daily_transaction_summary_by_source_df.write\
                                          .format("delta")\
                                          .mode("overwrite")\
                                          .saveAsTable("workspace.banking_gold.gold_daily_transaction_summary_by_source")

In [0]:
display(spark.table("workspace.banking_gold.gold_daily_transaction_summary_by_source").count())

In [0]:
gold_source_transactions_df.count()

In [0]:
# Understand customer transaction distribution
gold_source_transactions_df.groupBy(
    "customer_id","status"
).count().orderBy(
     "customer_id","status"
).show(50)

In [0]:
# Create the customer-level Gold aggregation
gold_daily_customer_summary_df = gold_source_transactions_df.groupBy(
    "transaction_date",
    "customer_id",
    "status"
).agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
)

In [0]:
gold_daily_customer_summary_df.orderBy(
    "transaction_date",
    "customer_id",
    "status"
).show()

In [0]:
gold_daily_customer_summary_df.agg(
    F.sum("transaction_count").alias("transaction_count"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
gold_daily_customer_summary_df.write\
                              .format("delta")\
                              .mode("overwrite")\
                              .saveAsTable("workspace.banking_gold.gold_daily_customer_summary")

In [0]:
display(spark.table("workspace.banking_gold.gold_daily_customer_summary").count())

In [0]:
# Daily Transaction Summary by Customer Segment

silver_customers_df = spark.table("workspace.banking_silver.silver_customers")

In [0]:
silver_customers_df.show()

In [0]:
# Join transactions with customer information
gold_segment_transactions_df = gold_source_transactions_df.join(
    silver_customers_df.select("customer_id",
                               "customer_segment"),
    "customer_id",
    "inner"
)

In [0]:
gold_segment_transactions_df.count()

In [0]:
# Validate customer segments
gold_segment_transactions_df.groupBy("customer_segment","status").count()\
                            .orderBy("customer_segment","status").show()

In [0]:
gold_daily_segment_summary_df = gold_segment_transactions_df.groupBy(
    "transaction_date",
    "customer_segment",
    "status"
).agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
)

In [0]:
gold_daily_segment_summary_df.orderBy(
    "transaction_date",
    "customer_segment",
    "status"
).show(50)

In [0]:
gold_daily_segment_summary_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
# Save the Gold table
gold_daily_segment_summary_df.write\
                             .format("delta")\
                             .mode("overwrite")\
                             .saveAsTable("workspace.banking_gold.gold_daily_segment_summary")

In [0]:
display(spark.table("workspace.banking_gold.gold_daily_segment_summary").count())

In [0]:
silver_accounts_df = spark.table("workspace.banking_silver.silver_accounts")

silver_accounts_df.select(
    "account_id",
    "account_type"
).show()

In [0]:
# Join transactions with account information
gold_account_transactions_df = gold_source_transactions_df.join(
    silver_accounts_df.select("account_id","account_type"),
    "account_id",
    "inner"
)

In [0]:
gold_account_transactions_df.count()

In [0]:
# Validate account-type and status distribution
gold_account_transactions_df.groupBy(
    "account_type",
    "status"
).count().orderBy(
    "account_type",
    "status"
).show()

In [0]:
# Create the Gold aggregation
gold_daily_account_type_summary_df = gold_account_transactions_df.groupBy(
    "transaction_date",
    "account_type",
    "status"
).agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
)

In [0]:
gold_daily_account_type_summary_df.orderBy(
    "transaction_date",
    "account_type",
    "status"
).show(50)

In [0]:
gold_daily_account_type_summary_df.agg(
    F.count("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
gold_daily_account_type_summary_df.agg(
    F.sum("transaction_count").alias("total_transactions")
).show()

gold_daily_account_type_summary_df.agg(
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
# Save the Gold table

gold_daily_account_type_summary_df.write\
                             .format("delta")\
                             .mode("overwrite")\
                             .saveAsTable("workspace.banking_gold.gold_daily_account_type_summary")


In [0]:
display(spark.table("workspace.banking_gold.gold_daily_account_type_summary").count())

In [0]:
# Check transaction-type distribution
gold_source_transactions_df.groupBy(
    "transaction_type"
).count().orderBy("transaction_type").show()

In [0]:
# Check transaction type and status distribution
gold_source_transactions_df.groupBy(
    "transaction_type",
    "status"
).count().orderBy("transaction_type","status").show()

In [0]:
# Create the Gold aggregation
gold_daily_transaction_type_summary_df = gold_source_transactions_df.groupBy(
    "transaction_date",
    "transaction_type",
    "status"
).agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
)

In [0]:
gold_daily_transaction_type_summary_df.orderBy(
    "transaction_date",
    "transaction_type",
    "status"
).show(50)

In [0]:
gold_daily_transaction_type_summary_df.agg(
    F.sum("transaction_count").alias("transaction_count"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
# save to delta table
gold_daily_transaction_type_summary_df.write\
                                      .format("delta")\
                                      .mode("overwrite")\
                                      .saveAsTable("workspace.banking_gold.gold_daily_transaction_type_summary")


In [0]:
display(spark.table("workspace.banking_gold.gold_daily_transaction_type_summary").count())

In [0]:
# Daily Business KPI Summary
gold_source_transactions_df.select(
    "transaction_date",
    "transaction_id",
    "amount",
    "status"
).show(10)

In [0]:
# Calculate successful transaction KPIs
successful_kpi_df = gold_source_transactions_df.groupBy(
    "transaction_date"
).agg(
    F.sum(
        F.when(F.col("status") == "SUCCESS",1).otherwise(0)
    ).alias("successful_count"),

    F.sum(
        F.when(F.col("status")=="SUCCESS",F.col("amount")).otherwise(0)
    ).alias("sucessful_amount")
)

successful_kpi_df.orderBy("transaction_date").show(50)

In [0]:
successful_kpi_df = successful_kpi_df.withColumnRenamed(
    "sucessful_amount", "successful_amount"
)

In [0]:
successful_kpi_df.printSchema()

In [0]:
# Calculate failed transaction KPIs

failed_kpi_df = gold_source_transactions_df.groupBy(
    "transaction_date"
).agg(
    F.sum(
        F.when(F.col("status") == "FAILED",1).otherwise(0)
    ).alias("failed_count"),

    F.sum(
        F.when(F.col("status") == "FAILED",F.col("amount")).otherwise(0)
    ).alias("failed_amount")
)

failed_kpi_df.orderBy("transaction_date").show(50)

In [0]:
# Join the two KPI DataFrames
daily_kpi_df = successful_kpi_df.join(
    failed_kpi_df,
    "transaction_date",
    "inner"
)

daily_kpi_df.orderBy("transaction_date").show(50)

In [0]:
# Calculate pending transaction KPIs
pending_kpi_df = gold_source_transactions_df.groupBy(
    "transaction_date"
).agg(
    F.sum(
        F.when(F.col("status") == "PENDING",1).otherwise(0)
    ).alias("pending_count"),

    F.sum(
        F.when(F.col("status") == "PENDING",F.col("amount")).otherwise(0)
    ).alias("pending_amount")
)

pending_kpi_df.orderBy("transaction_date").show(50)

In [0]:
# Calculate reversed transaction KPIs
reversed_kpi_df = gold_source_transactions_df.groupBy(
    "transaction_date"
).agg(
    F.sum(
        F.when(F.col("status") == "REVERSED",1).otherwise(0)
    ).alias("reversed_count"),

    F.sum(
        F.when(F.col("status") == "REVERSED",F.col("amount")).otherwise(0)
    ).alias("reversed_amount")
)

reversed_kpi_df.orderBy("transaction_date").show(50)

In [0]:
# Combine all four KPI DataFrames
daily_kpi_df = successful_kpi_df.join(
    failed_kpi_df,
    "transaction_date",
    "inner"
).join(
    pending_kpi_df,
    "transaction_date",
    "inner"
).join(
    reversed_kpi_df,
    "transaction_date",
    "inner"
)

daily_kpi_df.orderBy("transaction_date").show(50)

In [0]:
# Add total transaction count and total amount

daily_kpi_df = daily_kpi_df.withColumn(
    "total_transactions",
    F.col("successful_count") +
    F.col("failed_count") +
    F.col("pending_count") +
    F.col("reversed_count")
).withColumn(
    "total_amount",
    F.col("successful_amount") +
    F.col("failed_amount") +
    F.col("pending_amount") +
    F.col("reversed_amount")
)

daily_kpi_df.orderBy("transaction_date").show(50)

In [0]:
# Calculate success rate
daily_kpi_df = daily_kpi_df.withColumn(
    "success_rate",
    F.round(
        (F.col("successful_count")/F.col("total_transactions")) * 100,2
    )
)

In [0]:
daily_kpi_df.select(
    "transaction_date",
    "total_transactions",
    "successful_count",
    "success_rate"
).show(50)

In [0]:
# Validate the complete KPI dataset

daily_kpi_df.agg(
    F.sum("total_transactions").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount"),
    F.sum("successful_count").alias("successful_transactions"),
    F.sum("failed_count").alias("failed_transactions"),
    F.sum("pending_count").alias("pending_transactions"),
    F.sum("reversed_count").alias("reversed_transactions")
).show()

In [0]:
daily_kpi_df.write\
            .format("delta")\
            .mode("overwrite")\
            .saveAsTable("workspace.banking_gold.gold_daily_business_kpis")

In [0]:
display(spark.table("workspace.banking_gold.gold_daily_business_kpis").count())

In [0]:
spark.sql("""SHOW TABLES IN workspace.banking_gold""").show(50)

In [0]:
# Display complete table names

spark.sql("""SHOW TABLES IN workspace.banking_gold""").select("tableName").show(50)

In [0]:
tables_name = [
    row.tableName
    for row in spark.sql("""SHOW TABLES IN workspace.banking_gold""").collect()
]

print(tables_name)


In [0]:
for table_name in tables_name:
    count = spark.table(f"workspace.banking_gold.{table_name}").count()

    print(table_name,":",count)
      

In [0]:
# Verify Gold table schemas
spark.table("workspace.banking_gold.gold_daily_transaction_summary").printSchema()

In [0]:
spark.table("workspace.banking_gold.gold_daily_transaction_summary_by_source").printSchema()

In [0]:
spark.table("workspace.banking_gold.gold_daily_customer_summary").printSchema() 

In [0]:
spark.table("workspace.banking_gold.gold_daily_segment_summary").printSchema()

In [0]:
spark.table("workspace.banking_gold.gold_daily_account_type_summary").printSchema()

In [0]:
spark.table("workspace.banking_gold.gold_daily_transaction_type_summary").printSchema()

In [0]:
spark.table("workspace.banking_gold.gold_daily_business_kpis").printSchema()

In [0]:
spark.table("workspace.banking_gold.gold_daily_transaction_summary").groupBy("status").count().show()

In [0]:
spark.table(
    "workspace.banking_gold.gold_daily_transaction_summary"
).groupBy("transaction_type").count().show()

In [0]:
# Validate transaction count reconciliation.
gold_df = spark.table(
    "workspace.banking_gold.gold_daily_transaction_summary"
)

gold_df.agg(
    F.sum("transaction_count").alias("gold_transaction_count")
).show()

In [0]:
# Validate total transaction amount
gold_df.agg(
    F.sum("total_amount").alias("gold_total_amount")
).show()

In [0]:
# Validate Dataset 2 – Source summary
spark.table("workspace.banking_gold.gold_daily_transaction_summary_by_source"
            ).groupBy("source_system").count().show()

In [0]:
# Validate transaction count by source system.
source_gold_df = spark.table("workspace.banking_gold.gold_daily_transaction_summary_by_source")

source_gold_df.groupBy(
    "source_system"
).agg(
    F.sum("transaction_count").alias("total_transactions")
).show()

In [0]:
# Validate total amount by source
source_gold_df.agg(
    F.sum("total_amount").alias("gold_source_total_amount")
).show()

In [0]:
# Dataset 3 – Customer summary
customer_gold_df = spark.table("workspace.banking_gold.gold_daily_customer_summary")

customer_gold_df.select("customer_id").distinct().count()

In [0]:
# Validate transaction count by customer
customer_gold_df.groupBy("customer_id").agg(
    F.sum("transaction_count").alias("total_transactions")
).orderBy("customer_id").show()

In [0]:
# Validate customer-wise total amount
customer_gold_df.agg(
    F.sum("total_amount").alias("customer_total_amount")
).show()

In [0]:
# Dataset 4 – Customer segment summary

segment_gold_df  = spark.table("workspace.banking_gold.gold_daily_segment_summary")

segment_gold_df.groupBy("customer_segment").agg(
    F.sum("transaction_count").alias("total_transactions")
).show()


In [0]:
# Validate segment-wise total amount
segment_gold_df.agg(
    F.sum("total_amount").alias("segment_total_amount")
).show()

In [0]:
# Dataset 5 – Account type summary
account_gold_df = spark.table("workspace.banking_gold.gold_daily_account_type_summary")

account_gold_df.groupBy("account_type").agg(
    F.sum("transaction_count").alias("total_transactions")
).show()

In [0]:
# Validate account-type total amount
account_gold_df.agg(
    F.sum("total_amount").alias("account_total_amount")
).show()

In [0]:
# Dataset 6 – Transaction type summary
transaction_type_gold_df = spark.table("workspace.banking_gold.gold_daily_transaction_type_summary")

transaction_type_gold_df.groupBy("transaction_type").agg(
    F.sum("transaction_count").alias("total_transactions")
).show()

In [0]:
# Validate transaction-type total amount
transaction_type_gold_df.agg(
    F.sum("total_amount").alias("transaction_type_total_amount")
).show()

In [0]:
# Dataset 7 – Business KPIs
kpi_gold_df = spark.table("workspace.banking_gold.gold_daily_business_kpis")

kpi_gold_df.agg(
    F.sum("total_transactions").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
# Validate status-wise transaction counts
kpi_gold_df.agg(
    F.sum("successful_count").alias("successful"),
    F.sum("failed_count").alias("failed"),
    F.sum("pending_count").alias("pending"),
    F.sum("reversed_count").alias("reversed")
).show()

In [0]:
# Validate status-wise transaction amounts
kpi_gold_df.agg(
    F.sum("successful_amount").alias("successful_amount"),
    F.sum("failed_amount").alias("failed_amount"),
    F.sum("pending_amount").alias("pending_amount"),
    F.sum("reversed_amount").alias("reversed_amount")
).show()

In [0]:
silver_df = spark.table(
    "workspace.banking_silver.silver_transactions"
)

silver_df.groupBy("status").agg(
    F.sum("amount").alias("total_amount")
).orderBy("status").show()

In [0]:
# Validate success rate
kpi_gold_df.filter(
    F.col("success_rate") != 
    F.round(
        F.col("successful_count") / F.col("total_transactions")*100,2
    )
).show()

In [0]:
gold_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_transaction_amount")
).show()

In [0]:
source_gold_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_transaction_amount")
).show()

In [0]:
customer_gold_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
segment_gold_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
account_gold_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
transaction_type_gold_df.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
kpi_gold_df.agg(
    F.sum("total_transactions").alias("total_transactions"),
    F.sum("total_amount").alias("total_amount")
).show()

In [0]:
# Transaction count reconciliation

silver_count = spark.table("workspace.banking_silver.silver_transactions").count()

quarantine_count = spark.table("workspace.banking_quarantine.quarantine_transactions").count()

gold_count = spark.table("workspace.banking_gold.gold_daily_transaction_summary").agg(
    F.sum("transaction_count")
).first()[0]

print("Silver transactions:", silver_count)
print("Quarantined transactions:", quarantine_count)
print("Gold transactions:", gold_count)
print("Total accounted transactions:", silver_count + quarantine_count)

In [0]:
# Event count reconciliation
silver_event_count = spark.table("workspace.banking_silver.silver_events").count()

quarantine_event_count = spark.table("workspace.banking_quarantine.quarantine_events").count()


print("Silver events:",silver_event_count)
print("Quarantined events:", quarantine_event_count)
print("Total accounted events:",silver_event_count + quarantine_event_count)

In [0]:
# Customer count reconciliation

silver_customer_count = spark.table("workspace.banking_silver.silver_customers").count()

quarantine_customer_count = spark.table("workspace.banking_quarantine.quarantine_customers").count()

print("Silver customers:", silver_customer_count)
print("Quarantined customers:", quarantine_customer_count)
print(
    "Total accounted customers:",
    silver_customer_count + quarantine_customer_count
)

In [0]:
# Account count reconciliation
silver_account_count = spark.table("workspace.banking_silver.silver_accounts").count()

quarantine_account_count = spark.table("workspace.banking_quarantine.quarantine_accounts").count()

print("Silver accounts:", silver_account_count)
print("Quarantined accounts:", quarantine_account_count)
print(
    "Total accounted accounts:",
    silver_account_count + quarantine_account_count
)

In [0]:
# End-to-end amount reconciliation
silver_total_amount = spark.table("workspace.banking_silver.silver_transactions").agg(
    F.sum("amount")
).first()[0]

gold_total_amount = spark.table("workspace.banking_gold.gold_daily_transaction_summary").agg(
    F.sum("total_amount")
).first()[0]

print("Silver total amount:", silver_total_amount)
print("Gold total amount:", gold_total_amount)
print("Amount matched:", silver_total_amount == gold_total_amount)


In [0]:
# Validate transaction status distribution

print("SILVER STATUS DISTRIBUTION")

spark.table(
    "workspace.banking_silver.silver_transactions"
).groupBy("status").agg(
    F.count("*").alias("transaction_count")
).orderBy("status").show()

print("GOLD STATUS DISTRIBUTION")

spark.table(
    "workspace.banking_gold.gold_daily_transaction_summary"
).groupBy("status").agg(
    F.sum("transaction_count").alias("transaction_count")
).orderBy("status").show()  

In [0]:
# Validate transaction type distribution

print("SILVER TRANSACTION TYPE DISTRIBUTION")

spark.table(
    "workspace.banking_silver.silver_transactions"
).groupBy("transaction_type").agg(
    F.count("*").alias("transaction_count")
).orderBy("transaction_type").show()

print("GOLD TRANSACTION TYPE DISTRIBUTION")

spark.table(
    "workspace.banking_gold.gold_daily_transaction_type_summary"
).groupBy("transaction_type").agg(
    F.sum("transaction_count").alias("transaction_count")
).orderBy("transaction_type").show()

In [0]:
# Validate source system distribution

print("SILVER SOURCE DISTRIBUTION")

spark.table(
    "workspace.banking_silver.silver_transactions"
).groupBy("source_system").count().orderBy("source_system").show()

print("GOLD SOURCE DISTRIBUTION")

spark.table(
    "workspace.banking_gold.gold_daily_transaction_summary_by_source"
).groupBy("source_system").agg(
    F.sum("transaction_count").alias("transaction_count")
).orderBy("source_system").show()

In [0]:
# Validate customer segment distribution
print("SILVER CUSTOMER SEGMENT DISTRIBUTION")

spark.table(
    "workspace.banking_silver.silver_transactions"
).join(
    spark.table("workspace.banking_silver.silver_customers"),
    "customer_id"
).groupBy("customer_segment").count().orderBy(
    "customer_segment"
).show()

print("GOLD CUSTOMER SEGMENT DISTRIBUTION")

spark.table(
    "workspace.banking_gold.gold_daily_segment_summary"
).groupBy("customer_segment").agg(
    F.sum("transaction_count").alias("transaction_count")
).orderBy("customer_segment").show()

In [0]:
# Validate account type distribution
print("SILVER ACCOUNT TYPE DISTRIBUTION")

spark.table(
    "workspace.banking_silver.silver_transactions"
).join(
    spark.table("workspace.banking_silver.silver_accounts"),
    "account_id"
).groupBy("account_type").count().orderBy(
    "account_type"
).show()

print("GOLD ACCOUNT TYPE DISTRIBUTION")

spark.table(
    "workspace.banking_gold.gold_daily_account_type_summary"
).groupBy("account_type").agg(
    F.sum("transaction_count").alias("transaction_count")
).orderBy("account_type").show()

In [0]:
# Validate daily transaction counts
print("SILVER DAILY TRANSACTION COUNT")

spark.table(
    "workspace.banking_silver.silver_transactions"
).withColumn(
    "transaction_date",
    F.to_date("transaction_timestamp")
).groupBy("transaction_date").count().orderBy(
    "transaction_date"
).show()

print("GOLD DAILY TRANSACTION COUNT")

spark.table(
    "workspace.banking_gold.gold_daily_transaction_summary"
).groupBy("transaction_date").agg(
    F.sum("transaction_count").alias("transaction_count")
).orderBy("transaction_date").show()

In [0]:
%sql
SHOW TABLES IN workspace.banking_gold;

In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_business_kpis;

In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_transaction_summary;

In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_transaction_summary_by_source;


In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_customer_summary;

In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_segment_summary;

In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_account_type_summary;

In [0]:
%sql
DESCRIBE TABLE workspace.banking_gold.gold_daily_transaction_type_summary;

In [0]:
%sql
-- Export the business KPI dataset
SELECT * FROM workspace.banking_gold.gold_daily_business_kpis
ORDER BY transaction_date;